# **Hands-on Session 2: Downloading multiple model outputs and comparison of the lake model outputs with observations**

In this hands-on session, we will compare Lake Erie Operational Forecast System (LEOFS) model output with buoy observations and, optionally, satellite-derived lake surface temperature from GLSEA.

In the Hands-on Session 1, we worked with a single model output file. Here, we extend that workflow  to a short time series by downloading multiple hourly model files. By the end of the guided portion, you will be able to:
1.   Download multiple hourly GLOFS/LEOFS NetCDF files from NOAA’s public archive
2.   Download and read buoy observations from the National Data Buoy Center
3. Find the model grid node closest to a buoy location
4. Extract a model lake surface temperature time series
5. Plot and interpret model–observation differences


Also see [SessionII_Handson_1.ipynb](https://colab.research.google.com/drive/1Qzf5vxQpBC1S85C-l3i3kPUEVHt2M65N?usp=drive_link) for a basic workflow to download, read, and visualize a LEOFS output.

* **Estimated Time:** 1 hour 45 mins.
* **Prerequisites:** Basic Python plus some familiarity with NumPy/Matplotlib. No prior [FVCOM](https://github.com/FVCOM-GitHub/FVCOM)/GLOFS experience required.
* **What you'll need:** Python Environment (e.g., Google Colab, Jupyter Notebook)


| Part | Topic | Approx. time |
|------|-------|--------------|
| **Part 1** | Set up environment and download multiple LEOFS/GLOFS files | 20 min |
| **Part 2** | Download and inspect buoy observations | 15 min |
| **Part 3** | Find nearest model node and extract model time series  | 25 min |
| **Part 4** | Compare model and buoy lake surface temperature  | 15 min |
| **Optional** | Add GLSEA satellite data or try additional comparisons | 30 min |

# **Part 1: Set up environment and download multiple LEOFS/GLOFS files**

In the first hands-on session, we focused on a single timeslice. This time, we will look into a timeseries data. GLOFS datastream provides one time slice per file. Therefore, if you'd like to look into temporal changes, you will have to download multiple files.

## **1.1 Import libraries**


In [ ]:
# Install required packages.
# boto3 is used to access NOAA files on AWS S3.
! pip install boto3
! pip install botocore
from datetime import datetime, timedelta
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
import pandas as pd
import re
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# set this False if you are using a temporary folder
use_google_drive = True

if use_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = Path("/content/drive/MyDrive/GL_env_data/handson2")
else:
    save_dir = Path("/content/GL_env_data/handson2")


# Create main folder.
save_dir.mkdir(parents=True, exist_ok=True)

print(f"Files will be saved in: {save_dir}")



## **1.2 Download 7-day long GLOFS outputs**
In this example, we are downloading data from June 1-7, 2025. You are welcome to explore another period. The process involves downloading hourly files (a total of 168 files) and takes a few minutes to complete.

**Note:** Download times may vary depending on network speed and other factors. If downloads take longer than expected, participants can reduce the date range to 2–3 days.

In this notebook, **GLOFS** refers to NOAA’s Great Lakes Operational Forecast System family.  
**LEOFS** is the Lake Erie member of that system, and it is the example system used here. This example downloads data for Lake Erie (leofs). If you'd like, you can also specify a system for another Great Lake.


| Abbreviation| Full System Name |
|-------------|--------------|
| leofs | Lake Erie Operational Forecast System |
| lmhofs | Lake Michigan-Huron Operational Forecast System |
| lsofs | Lake Superior Operational Forecast System |
| loofs | Lake Ontario Operational Forecast System |


In [ ]:
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# set start and end dates
start_date = datetime(2025,6,1)
end_date = datetime(2025,6,7)

# lake name
systemname='leofs'

# Create folder if it does not exist
os.makedirs(save_dir/'glofs', exist_ok=True)

# bucket name
bucket = "noaa-nos-ofs-pds"

current_date = start_date

# loop through the days
while current_date != end_date+timedelta(days=1):
 # print(current_date)

  yearstr=str(current_date.year).zfill(4)
  monstr=str(current_date.month).zfill(2)
  daystr=str(current_date.day).zfill(2)


  # loop thorugh each cycle (4 times daily, every 6 hours) and each nowcast hours (every hour)
  for cycle in range(0,24,6):
    for hour in range(6):
      key = systemname+"/netcdf/"+yearstr+"/"+monstr+"/"+daystr+"/"+systemname+".t"+str(cycle).zfill(2)+"z."+yearstr+monstr+daystr+".fields.n"+str(hour).zfill(3)+".nc"
      filename = systemname+".t"+str(cycle).zfill(2)+"z."+yearstr+monstr+daystr+".fields.n"+str(hour).zfill(3)+".nc"
      save_path = os.path.join(save_dir/'glofs', filename)
      print("Downloading "+key+"...")

      # Add retry mechanism for network errors
      max_retries = 3
      retries = 0
      while retries < max_retries:
          try:
              s3.download_file(bucket, key, save_path)
              break # If successful, break the retry loop
          except OSError as e:
              if e.errno == 107: # Check for "Transport endpoint is not connected"
                  retries += 1
                  print(f"Connection error: {e}. Retrying ({retries}/{max_retries})...")
                  time.sleep(5) # Wait for 5 seconds before retrying
              else:
                  raise # Re-raise other OSErrors
          except Exception as e:
              print(f"An unexpected error occurred: {e}")
              raise # Re-raise other exceptions
      else:
          print(f"Failed to download {key} after {max_retries} retries.")

  current_date=current_date + timedelta(days=1)


You can check that the downloaded files are stored in your Google Drive folder by navigating to the area in your Google Drive, or using a Unix syntax (ls).

In [ ]:
! ls /content/drive/MyDrive/GL_env_data/handson2/glofs -l


#**Part 2	Download and inspect buoy observations**

##**2.1 Download Western Lake Erie buoy data from the NDBC site**
Next, we will download buoy observation data from [the National Data Buoy Center (NDBC)](https://www.ndbc.noaa.gov/). This example uses an offshore buoy in Western Lake Erie (45005 - https://www.ndbc.noaa.gov/station_page.php?station=45005), but you are welcome to try other buoy stations.



In [ ]:
# Create folder if it does not exist
os.makedirs(save_dir/'buoy', exist_ok=True)

# Download data
! wget https://www.ndbc.noaa.gov/data/historical/stdmet/45005h2025.txt.gz
! gunzip 45005h2025.txt.gz
# and move to /content/drive/MyDrive/GL_env_data/handson2/buoy
os.system("mv 45005h2025.txt " + str(save_dir / "buoy"))



##**2.2 Read the buoy data**

The buoy data is in the ASCII format (text based, human readable). You can view it using a text editor if you'd like. In this example, we will read it using Pandas.

In [ ]:
# column names
cols = ["YY", "MM", "DD", "hh", "mm",
    "WDIR", "WSPD", "GST", "WVHT", "DPD", "APD", "MWD",
    "PRES", "ATMP", "WTMP", "DEWP", "VIS", "TIDE"]

# read data using Pandas' read_csv
df = pd.read_csv(save_dir/'buoy/45005h2025.txt',comment="#",
                 names=cols, sep='\\s+',skiprows=2, na_values=[99, 999])
# print the header
print("Below is the original data frame:")
print(df.head())
print(" ")

# set time index
# build a datetime column
df["datetime"] = pd.to_datetime(df[["YY", "MM", "DD", "hh", "mm"]].rename(
        columns={"YY": "year","MM": "month","DD": "day",
            "hh": "hour","mm": "minute"}))
df = df.drop(columns=["YY", "MM", "DD", "hh", "mm"])

# Optional: make datetime the index
df = df.set_index("datetime")

# print the header again. Now you can see the date & time as the index.
print("Below is the data frame where the date and time are used for index:")
print(df.head())

##**2.3 Make a timeseries plot of buoy lake surface temperature**

As you can see, there are a number of meteorological variables (WDIR, WSPD, GST....). You can check out what they mean in the NDBC's data description page (https://www.ndbc.noaa.gov/faq/measdes.shtml).

Here, we will be looking at the water surface temperature data (WTMP). Let's begin by making a quick timeseries plot.

Anything standing out to you? Does the data cover the whole year or not? What do you think the noisy feature comes from?

In [ ]:
# plot 'WTMP' using Pandas' plot function.
df['WTMP'].plot(ylabel="Water temperature (°C)")

#**Part 3:	Find nearest model node and extract model time series**
In order to compare the lake surface temperature from the model outputs (LEOFS) and the buoy observation, we will first need to extract the LEOFS data at the closest to the buoy location. You can check the coordinate (longitude and latitude) from [the NDBC site](https://www.ndbc.noaa.gov/station_page.php?station=45005). In case of 45005, it's 41.677 N 82.398 W (41°40'36" N 82°23'54" W).

We also have to concatenate the series of 7-day (168-hour) long LEOFS data files.

##**3.1 Create a list of all LEOFS data files**
The downloaded LEOFS files are not concatenated in time (one file per time slice). We will first create a list of all LEOFS data files, sorted in time.

In [ ]:
#import re
#from pathlib import Path
#import xarray as xr

# data directory where the LEOFS output files are stored.
data_dir = Path("/content/drive/MyDrive/GL_env_data/handson2/glofs")


# define a function to find multiple files
def sort_key(path):
    """
    Sort files like:
    leofs.t00z.20250701.fields.n000.nc
    by date, cycle hour, and n-hour.
    """
    m = re.search(r"leofs\.t(\d{2})z\.(\d{8})\.fields\.n(\d{3})\.nc", path.name)
    if m is None:
        return ("99999999", 99, 999)
    cycle_hour = int(m.group(1))
    yyyymmdd = m.group(2)
    n_hour = int(m.group(3))
    return (yyyymmdd, cycle_hour, n_hour)


# use the function to find all LEOFS files
files = sorted(data_dir.glob("leofs.t*z.*.fields.n*.nc"), key=sort_key)

print(f"Found {len(files)} files")



##**3.2 Find the nearest node of LEOFS to the buoy location**
Next, we will find the nearest node of LEOFS mesh to the buoy location. This allows us to extract the data at the specific node only, rather than the whole mesh data, saving significant amount of time for processing.

In [ ]:
import numpy as np


# buoy lon & lat
target_lon = -82.398
target_lat = 41.677

# example data
ds0=xr.open_dataset(data_dir/'leofs.t00z.20250601.fields.n000.nc')

lon = ds0['lon'].values
lat = ds0['lat'].values

# Convert target longitude if model longitude uses 0–360 convention
if np.nanmax(lon) > 180 and target_lon < 0:
    target_lon_model = target_lon % 360
else:
    target_lon_model = target_lon

# Approximate distance calculation, good enough for nearest-node lookup
dist2 = (((lon - target_lon_model) * np.cos(np.deg2rad(target_lat))) ** 2
    + (lat - target_lat) ** 2)

nearest_node = int(np.nanargmin(dist2))

print("Nearest node index:", nearest_node)
print("Nearest model lon:", float(lon[nearest_node])-360.)
print("Nearest model lat:", float(lat[nearest_node]))

##**3.3 Extract data from LEOFS**
We will extract data from the LEOFS files (sorted in time as in the 'files' list), just for the surface (siglay=0), and at the nearest node to the buoy location. While this saves a significant amount of time as opposed to reading data over the entire vertical (or sigma) layers and the entire node, it still takes some time to read all the 168 files. Estimated time for this process is 6-12 mins.

In [ ]:
# define a preprocess function to just extract the surface, and the nearest node
def preprocess(ds):
    """
    Keep only surface temp at the nearest node.
    This greatly reduces memory use before concatenation.
    """
    da = ds["temp"]

    # Select surface layer (siglay=0)
    da = da.isel(siglay=0)
    # ...and the nearest node
    da = da.isel(node=nearest_node)

    # Return as a Dataset so open_mfdataset can combine cleanly
    return da.to_dataset(name="temp_surface_point")


# open the LEOFS files

# open_mfdataset opens multuple files and combine into a dataset
ds_point = xr.open_mfdataset(
    # the list of files created earlier
    files,
    # xarray should combine in the listed order
    combine="nested",
    # concatenate in time
    concat_dim="time",
    # call the pre-process function before combining
    preprocess=preprocess,
    # sequential file opening, may try "True"
    parallel=False,
#    chunks={"time": 24},
    decode_times=True,
)

print(ds_point)

# quick plot
ds_point['temp_surface_point'].plot()


# **Part 4: Compare model and buoy lake surface temperature**

Finally, we will plot a timeseries of the lake surface temperature from the LEOFS model outputs, and the buoy observations. The comparison is for the 7-day period for which we downloaded the LEOFS model outputs.

In [ ]:
#import matplotlib.pyplot as plt

fig, ax=plt.subplots(1,1)

# plot LEOFS model output extracted at the nearest node to the buoy
ax.plot(ds_point.time,ds_point['temp_surface_point'],'-', label='LEOFS')



# plot the 45005 buoy data
ax.plot(df.index, df['WTMP'], '-', label="buoy 45005")
ax.set_ylabel("Water temperature (°C)")

# focus on the 7-day period
ax.set_xlim(ds_point.time[0],ds_point.time[-1])

# set y range
#ax.set_ylim(ds_point['temp_surface_point'].min()-1,ds_point['temp_surface_point'].max()+1)
ax.set_ylim(12,19)

# add legend
ax.legend()


fig.autofmt_xdate()


You can see the gradual warming of lake surface temperatture, as the time progresses from early to mid summer. You can also see the diurnal (day-night) cycle. Note that the times are in UTC (Coordinated Universal Time).  

Because LEOFS/FVCOM uses an unstructured grid, the nearest model node may not be exactly at the buoy location.
The comparison is therefore between the buoy and the closest model grid point, not a perfectly colocated measurement.

### Checkpoint: interpret the comparison

Take 2–3 minutes to discuss with a neighbor:

1. Does LEOFS capture the overall warming trend during this period?
2. Does LEOFS capture the day–night temperature cycle?
3. Is there a consistent warm or cold bias?
4. Why might the buoy observations have a “stair-step” appearance?
5. What are possible reasons for differences between a model grid point and a buoy measurement?

You are welcome to explore different visualizations, such as adding more time ticks, using different color schemes, comparing at another buoy location, and calculating differences.

#**Optional:	Add GLSEA data**


If time allows, we will add GLSEA satellite-derived lake surface temperature to the comparison.

GLSEA provides daily lake surface temperature estimates on a regular latitude–longitude grid. Unlike the buoy, which is a point measurement, GLSEA represents a gridded satellite-based product. This makes it useful for spatial context, but it may differ from buoy observations because of cloud cover, spatial averaging, satellite retrieval assumptions, and timing.

Through this optional analysis, we can evaluate the differences among the three datasets.

First, we will begin by downloading GLSEA data for the 7-day period.

In [ ]:
import urllib

# set up a directory to store GLSEA files
glsea_dir = save_dir / "glsea"
glsea_dir.mkdir(parents=True, exist_ok=True)

url_glsea = "https://apps.glerl.noaa.gov/thredds/fileServer/glsea_nc_3"

current_date = start_date

while current_date <= end_date:
    yearstr = f"{current_date.year:04d}"
    monstr = f"{current_date.month:02d}"
    doystr = current_date.strftime("%j")

    fname = f"{yearstr}_{doystr}_glsea_sst.nc"
    url_file = f"{url_glsea}/{yearstr}/{monstr}/{fname}"
    save_path = glsea_dir / fname

    if save_path.exists():
        print(f"Already exists, skipping: {fname}")
    else:
        print(f"Downloading {fname}")
        urllib.request.urlretrieve(url_file, save_path)

    current_date += timedelta(days=1)

Similarly to the LEOFS outputs, we will read data at a point closest to the buoy, and append the data in time. If you would like to plot the GLSEA lake surface temperature on a map, please refer to [supplemental_GLSEA.ipynb](https://colab.research.google.com/drive/1y5JhOoamksgJ6cXIXQo1NN7yLkeGqi5Q?usp=sharing). Because GLSEA is on a regular grid and has longitude (lon) and latitude (lat) as dimensions. Extracting the nearest point is much simpler than that for LEOFS (or FVCOM).

In [ ]:
# find files in the folder
file_pattern = "*_glsea_sst.nc"
data_dir=Path(save_dir/'glsea')
files = sorted(data_dir.glob(file_pattern))

# open and concatenate files
ds_glsea = xr.open_mfdataset(files, combine="by_coords")

# extract data at the closest point to the buoy location
#(target_lon & target lat are set earlier)
ds_glsea_point = ds_glsea.sel(lat=target_lat,lon=target_lon, method="nearest")

## print
#ds_glsea_point.sst.values

# make a quick plot
ds_glsea_point.sst.plot()


Now, we will add this data to the timeseries plot we created for lake surface temperature at 45005 buoy location.

In [ ]:

# create a plot
fig, ax=plt.subplots(1,1)

# plot LEOFS model output extracted at the nearest node to the buoy
ax.plot(ds_point.time,ds_point['temp_surface_point'],'-', label='LEOFS')

# add GLSEA data
ax.plot(ds_glsea_point.time,ds_glsea_point['sst'],'o',label='GLSEA')

# add the 45005 buoy data
ax.plot(df.index, df['WTMP'], '-', label="buoy 45005")
ax.set_ylabel("Water temperature (°C)")

# focus on the 7-day period
ax.set_xlim(ds_point.time[0],ds_point.time[-1])

# set y range
#ax.set_ylim(ds_point['temp_surface_point'].min()-1,ds_point['temp_surface_point'].max()+1)
ax.set_ylim(12,19)

# add legend
ax.legend()


fig.autofmt_xdate()

What do you observe in your comparison? There are a lot of interesting considerations: What difference do you see in GLSEA satellite observation from buoy observation? Which of GLSEA or buoy do you think is useful for verifying the model outputs?

You are welcome to refer to further readings below, and discuss with other participants and your colleagues.


##**Further readings and data sources**

* FVCOM Github repository: https://github.com/FVCOM-GitHub/FVCOM

* FVCOM User manual: https://etchellsfleet27.com/wp-content/uploads/2020/06/FVCOM_User_Manual_v3.1.6.pdf

*   SCHISM utility scripts by James Kessler at NOAA Great Lakes Environmental Research Lab: https://github.com/NOAA-GLERL/SCHISM_grid_utils. Can be adaptable for other unstructured mesh model outputs, such as FVCOM.


* NOAA National Ocean Service Operational Forecast Systems: https://tidesandcurrents.noaa.gov/models.html

* NOAA National Data Buoy Center:
  https://www.ndbc.noaa.gov/

* NOAA CoastWatch Great Lakes Regional Node: https://coastwatch.glerl.noaa.gov/

* NOAA GLERL Great Lakes Surface Environmental Analysis:
  https://coastwatch.glerl.noaa.gov/glsea/

* NOAA NOS OFS public data archive on AWS:
https://registry.opendata.aws/noaa-ofs/